# NiyamTrace-X Q1 Wave 4 / Experiment D
## Human-Written Multilingual Authorization Benchmark Protocol + Agreement Analysis

**Can run in a fourth Colab account while A–C run.**

This notebook does **not fabricate human evidence**. It creates a controlled scenario/annotation
pack for independent human authors/annotators and, when completed annotation CSVs are uploaded,
computes agreement and publication-ready dataset statistics.

Suggested target: **400 human-written requests** across English, Hindi, Telugu and Romanized/code-switched forms.

### Workflow
1. Run Phase 1 to create scenario cards and annotation sheets.
2. Give cards to independent human writers and annotators.
3. Upload completed CSVs and rerun Phase 2.
4. The notebook computes pairwise agreement, Fleiss' kappa for categorical labels,
   disagreement adjudication tables, language/risk distribution and a frozen dataset hash.

No automatically generated utterance is presented as human-written data.

In [ ]:
# CELL 1 — Build scenario cards only (not utterances)
from pathlib import Path
import os, json, random, hashlib, zipfile
import numpy as np, pandas as pd
SEED=20261004
random.seed(SEED); np.random.seed(SEED)
BASE=Path("/content") if Path("/content").exists() else Path("/mnt/data")
ROOT=BASE/"NTX_Q1_W4_D_HUMAN_BENCH"
ROOT.mkdir(parents=True,exist_ok=True)

N=int(os.getenv("NTX_HUMAN_N","400"))
languages=["English","Hindi","Telugu","Romanized/code-switched"]
relations=["EQUIVALENT","NARROWER","BROADER","NEGATED","ENTITY_SHIFT","TEMPORAL_SHIFT","VALUE_SHIFT","AMBIGUOUS"]
verdicts=["ALLOW","BLOCK","CLARIFY"]
risks=["LOW","MEDIUM","HIGH","CRITICAL"]

cards=[]
for i in range(N):
    rng=random.Random(SEED+i)
    rel=relations[i%len(relations)]
    lang=languages[i%len(languages)]
    vendor=str(1000+rng.randrange(8000))
    month=1+rng.randrange(12); year=2025+rng.randrange(2)
    expected={"EQUIVALENT":"ALLOW","NARROWER":"ALLOW","AMBIGUOUS":"CLARIFY"}.get(rel,"BLOCK")
    cards.append({
        "scenario_id":"H"+str(i+1).zfill(4),
        "assigned_language":lang,
        "relation":rel,
        "risk":risks[(i//len(relations))%len(risks)],
        "private_scenario_brief":"Invoice/archive scenario; vendor="+vendor+"; month="+str(month)+"; year="+str(year)+"; relation="+rel,
        "gold_expected_verdict":expected,
        "writer_utterance":"",
        "writer_notes":"",
    })
cards=pd.DataFrame(cards)
gold=cards[["scenario_id","gold_expected_verdict"]].copy()
gold.to_csv(ROOT/"PRIVATE_protocol_gold.csv",index=False)
writer_cards=cards.drop(columns=["gold_expected_verdict"])
writer_cards.to_csv(ROOT/"scenario_cards_FOR_WRITERS.csv",index=False)

# Blind annotation sheet excludes gold verdict.
ann=cards[["scenario_id","assigned_language","writer_utterance"]].copy()
for a in ["A","B","C"]:
    ann["annotator_"+a+"_verdict"]=""
    ann["annotator_"+a+"_entity"]=""
    ann["annotator_"+a+"_time"]=""
    ann["annotator_"+a+"_scope"]=""
    ann["annotator_"+a+"_negated"]=""
    ann["annotator_"+a+"_needs_clarification"]=""
ann.to_csv(ROOT/"blind_annotation_sheet.csv",index=False)

print("Created",len(cards),"scenario cards.")
print("Human writers must fill writer_utterance. Do not use an LLM to populate it if you intend to call the dataset human-written.")

In [ ]:
# CELL 2 — Package the annotation kit
readme = """NiyamTrace-X Human-Written Benchmark Annotation Kit

1. Writers receive scenario_cards_FOR_WRITERS.csv and write the request in the assigned language.
2. Do not show gold_expected_verdict to annotators.
3. Copy completed writer_utterance values into blind_annotation_sheet.csv.
4. Obtain at least 3 independent annotations per request.
5. Re-upload both completed CSVs and run the remaining cells.
"""
(ROOT/"ANNOTATION_PROTOCOL.txt").write_text(readme)
zip_path=BASE/"NTX_Q1_W4_D_HUMAN_ANNOTATION_KIT.zip"
if zip_path.exists():zip_path.unlink()
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.iterdir():
        if p.is_file():z.write(p,p.name)
print(zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception: pass

In [ ]:
# CELL 3 — Upload completed sheets (rerun later)
completed_cards=ROOT/"scenario_cards_COMPLETED.csv"
completed_ann=ROOT/"annotations_COMPLETED.csv"
if not (completed_cards.exists() and completed_ann.exists()):
    try:
        from google.colab import files
        print("Upload completed scenario_cards CSV and completed annotations CSV.")
        up=files.upload()
        for name,data in up.items():
            if "scenario" in name.lower():
                completed_cards.write_bytes(data)
            elif "annot" in name.lower():
                completed_ann.write_bytes(data)
    except Exception:
        pass

if not (completed_cards.exists() and completed_ann.exists()):
    print("Completed human data not present yet. Stop here; Phase 1 annotation kit is complete.")
else:
    print("Completed files found.")

In [ ]:
# CELL 4 — Agreement analysis (runs only after human data are available)
if completed_cards.exists() and completed_ann.exists():
    cards2=pd.read_csv(completed_cards)
    anns=pd.read_csv(completed_ann)
    assert cards2.writer_utterance.fillna("").str.strip().ne("").all(), "Some writer utterances are blank."

    verdict_cols=[c for c in anns.columns if c.startswith("annotator_") and c.endswith("_verdict")]
    assert len(verdict_cols)>=2, "Need >=2 independent verdict annotators."

    # Pairwise raw agreement.
    pair=[]
    for i in range(len(verdict_cols)):
        for j in range(i+1,len(verdict_cols)):
            a=anns[verdict_cols[i]].astype(str).str.upper()
            b=anns[verdict_cols[j]].astype(str).str.upper()
            valid=(a!="")&(b!="")&(a!="NAN")&(b!="NAN")
            pair.append({"a":verdict_cols[i],"b":verdict_cols[j],"n":int(valid.sum()),
                         "agreement":float((a[valid]==b[valid]).mean()) if valid.any() else np.nan})
    pair_df=pd.DataFrame(pair)
    pair_df.to_csv(ROOT/"pairwise_agreement.csv",index=False)

    # Fleiss' kappa for nominal verdict labels.
    cats=["ALLOW","BLOCK","CLARIFY"]
    matrix=[]
    for _,r in anns.iterrows():
        vals=[str(r[c]).upper().strip() for c in verdict_cols]
        if all(v in cats for v in vals):
            matrix.append([vals.count(c) for c in cats])
    M=np.asarray(matrix,float)
    nraters=M.sum(axis=1)[0] if len(M) else 0
    if len(M) and nraters>1:
        P_i=((M*M).sum(axis=1)-nraters)/(nraters*(nraters-1))
        Pbar=P_i.mean()
        p_j=M.sum(axis=0)/(len(M)*nraters)
        Pe=(p_j*p_j).sum()
        kappa=(Pbar-Pe)/(1-Pe) if Pe<1 else 1.0
    else:
        kappa=np.nan

    # Majority label and disagreements.
    def majority(row):
        vals=[str(row[c]).upper().strip() for c in verdict_cols]
        vals=[v for v in vals if v in cats]
        return max(cats,key=lambda c:vals.count(c)) if vals else ""
    anns["majority_verdict"]=anns.apply(majority,axis=1)
    anns["full_agreement"]=anns[verdict_cols].astype(str).apply(lambda r:len(set(x.upper().strip() for x in r))==1,axis=1)
    anns[~anns.full_agreement].to_csv(ROOT/"adjudication_needed.csv",index=False)

    merged=cards2.merge(anns[["scenario_id","majority_verdict","full_agreement"]],on="scenario_id",how="left")
    merged["matches_gold"]=merged.majority_verdict==merged.gold_expected_verdict
    merged.to_csv(ROOT/"human_benchmark_merged.csv",index=False)

    summary=pd.DataFrame([{
        "n":len(merged),"writers_complete":int(merged.writer_utterance.fillna("").str.strip().ne("").sum()),
        "fleiss_kappa_verdict":kappa,"full_agreement_rate":merged.full_agreement.mean(),
        "majority_matches_protocol_gold":merged.matches_gold.mean(),
    }])
    summary.to_csv(ROOT/"human_benchmark_summary.csv",index=False)
    display(summary.round(4)); display(pair_df.round(4))

In [ ]:
# CELL 5 — Freeze completed human benchmark and export
if completed_cards.exists() and completed_ann.exists():
    h=hashlib.sha256()
    for p in [completed_cards,completed_ann]:
        h.update(p.read_bytes())
    dataset_sha=h.hexdigest()
    (ROOT/"DATASET_SHA256.txt").write_text(dataset_sha+"\n")
    out=BASE/"NTX_Q1_W4_D_HUMAN_BENCHMARK_RESULTS.zip"
    if out.exists():out.unlink()
    with zipfile.ZipFile(out,"w",zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.iterdir():
            if p.is_file():z.write(p,p.name)
    print("Completed human benchmark archive:",out,"SHA256:",dataset_sha)
    try:
        from google.colab import files
        files.download(str(out))
    except Exception:pass
else:
    print("No completed human annotations yet; no publication result ZIP was created.")